# ISM 6564 — Assignment 4: does reading in order pay for itself?

**Northwind Outfitters, review triage.** Every product review is scored by a
sentiment classifier: negative reviews open a support ticket, positive ones are
sampled for marketing quotes. The classifier is last month's kind of model — a
bag of words — and the support team has found reviews it gets backwards. Your
job is to build the Week 4 alternative, measure where it helps and what it
costs, and recommend what Northwind should run.

The full brief, the rubric, and the submission list are on the **assignment
page in Canvas**. This notebook needs nothing else: the data loads itself from
the public course data repository.

How this notebook works:

- Cells marked **`GIVEN`** are provided. Run them; do not modify them.
- Cells marked **`TODO`** are yours. Each starts with `None` — replace it with
  your work.
- The cell after each TODO **checks** your work and prints a clear message
  while anything is missing, so the notebook always runs top to bottom. Do not
  delete the check cells.
- Italic text in a *Your answer* cell is a placeholder: replace it with your
  own writing. Written answers must be your own words (course AI policy).

---
## Setup and data

In [1]:
# GIVEN — imports, seeds, and where the data lives.
import random
import time
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression

SEED = 42

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

set_seed()
DATA = "https://raw.githubusercontent.com/prof-tcsmith/6564F26-DATA/main/week-04-sequence-models/data/"
SLICES = ["simple", "contrast", "scope_short", "scope_long"]
print(f"torch {torch.__version__} — everything here runs on an ordinary laptop CPU")

torch 2.11.0+cpu — everything here runs on an ordinary laptop CPU


In [2]:
# GIVEN — load the reviews. The split is fixed in the file: do not re-split it.
reviews = pd.read_csv(DATA + "reviews.csv")
train_df = reviews[reviews["split"] == "train"].reset_index(drop=True)
val_df = reviews[reviews["split"] == "val"].reset_index(drop=True)
test_df = reviews[reviews["split"] == "test"].reset_index(drop=True)

print(f"{len(reviews)} reviews: train {len(train_df)}, validation {len(val_df)}, test {len(test_df)}\n")
print("test reviews per slice:", test_df["slice"].value_counts()[SLICES].to_dict(), "\n")
for s in SLICES:
    row = test_df[test_df["slice"] == s].iloc[0]
    print(f"[{s:<11}] {'positive' if row['label'] == 1 else 'negative'}: {row['text']}")

4837 reviews: train 3164, validation 469, test 1204

test reviews per slice: {'simple': 273, 'contrast': 296, 'scope_short': 336, 'scope_long': 299} 

[simple     ] positive: the companion app is responsive and clean
[contrast   ] negative: ordered this in june the mounting bracket lines up perfectly but the keys started sticking almost immediately model number lc four seven five
[scope_short] positive: buying for a home office it is not true that the hinge went loose within a month updating this review later
[scope_long ] negative: bought during the autumn sale it is not true that even allowing for the most recent firmware update compared with the previous model i owned the sound is surprisingly rich ordered through the app


Each review's `slice` says which kind of word order decides its label:

| `slice` | What it looks like |
|---|---|
| `simple` | one opinion, no tricks |
| `contrast` | *"A **but** B"* — the label follows **B**; every review also appears in the opposite order, with the opposite label |
| `scope_short` | a frame that may contain **not** (*"it is not true that…"*), then the opinion a few words later |
| `scope_long` | the same, with 15–20 filler words between the *not* and the opinion |

In [3]:
# GIVEN — scoring helpers. Every model's test results go into one table.
results = []

def slice_report(name, predictions, train_seconds, n_params):
    correct = predictions == test_df["label"].values
    row = {"model": name, "overall": correct.mean()}
    for s in SLICES:
        row[s] = correct[test_df["slice"].values == s].mean()
    row["train seconds"] = round(train_seconds, 2)
    row["learned numbers"] = n_params
    results.append(row)

def show_results():
    print(pd.DataFrame(results).set_index("model").round(4).to_string())

---
## Task 1 — The bag of words, and exactly where it fails

Two versions of the model Northwind runs today: TF-IDF over **single words**,
and TF-IDF over **single words plus word pairs** (bigrams: every two adjacent
words also count as a feature, so *not good* gets a column of its own). Both
are given — run them and read the table.

In [4]:
# GIVEN — the two bag-of-words models.
def fit_bag_of_words(ngram_range):
    t0 = time.perf_counter()
    vec = TfidfVectorizer(ngram_range=ngram_range)
    model = LogisticRegression(max_iter=2000).fit(vec.fit_transform(train_df["text"]), train_df["label"])
    seconds = time.perf_counter() - t0
    predictions = model.predict(vec.transform(test_df["text"]))
    n_params = model.coef_.size + model.intercept_.size
    return vec, model, predictions, seconds, n_params

words_vec, words_model, words_pred, words_secs, words_params = fit_bag_of_words((1, 1))
pairs_vec, pairs_model, pairs_pred, pairs_secs, pairs_params = fit_bag_of_words((1, 2))
slice_report("bag of words: single words", words_pred, words_secs, words_params)
slice_report("bag of words: + word pairs", pairs_pred, pairs_secs, pairs_params)
show_results()

                            overall  simple  contrast  scope_short  scope_long  train seconds  learned numbers
model                                                                                                         
bag of words: single words   0.6453     1.0    0.5000       0.5327      0.5920           0.10              289
bag of words: + word pairs   0.7533     1.0    0.9291       0.5357      0.5987           0.08             1350


In [5]:
# GIVEN — look closely at one contrast pair from the test set.
contrast = test_df[test_df["slice"] == "contrast"]
key = contrast["text"].map(lambda t: " ".join(sorted(t.split())))
pair = contrast[key == key.iloc[0]]
for _, row in pair.iterrows():
    p = words_model.predict_proba(words_vec.transform([row["text"]]))[0, 1]
    print(f"true: {'positive' if row['label'] == 1 else 'negative'} | single-words model P(positive) = {p:.4f}")
    print(f"   {row['text']}\n")

true: negative | single-words model P(positive) = 0.5505
   ordered this in june the mounting bracket lines up perfectly but the keys started sticking almost immediately model number lc four seven five

true: positive | single-words model P(positive) = 0.5505
   ordered this in june the keys started sticking almost immediately but the mounting bracket lines up perfectly model number lc four seven five



### Your answer — Task 1

The single words model gets exactly 0.5000 on contrast. Because bag of words ignores order, the orderings of a contrast review have identical words, so the model builds an identical feature vector for both and assigns them the identical probability, the printed pair above both get P(positive) = 0.5505. Since the two orderings always carry opposite true labels, one is always right and one always wrong, so accuracy is always 0.5.

Word pairs rescue contrast (0.9291) but leave scope_short and scope_long almost the same (0.5357 and 0.5987). That's because the decisive clause in a contrast review is beside the word "but," so a bigram like "but the" can capture which clause comes last. In the scope reviews, the opinion is separated from "not" by a handful of words, which is a lot more than the two word window a bigram can see, so no adjacent word feature has enough information to resolve the negation. 

---
## Task 2 — Build the model that reads in order

Build the Week 4 model: an **embedding** table (word ID → 64 numbers; the rows
start random and are learned from this task), an **LSTM** that reads the review
one word at a time and carries a 64-number memory, and the **head** — a linear
layer that turns the memory after the last word into two scores (negative,
positive). Two sizes, both 64 here: how many numbers describe a word, and how
many the memory carries. The words-to-IDs encoding and the training loop are
given — they are the same as in the lecture notebook's §5.

In [6]:
# GIVEN — words -> IDs, exactly as in the lecture notebook (vocabulary from TRAIN only).
tokenize = CountVectorizer().build_analyzer()
MAX_LEN = 50                                    # the longest review has 49 words

word_counts = Counter(w for text in train_df["text"] for w in tokenize(text))
vocab = {"<pad>": 0, "<unk>": 1}
for w, n in word_counts.items():
    if n >= 2:
        vocab[w] = len(vocab)

def encode(texts):
    rows = np.zeros((len(texts), MAX_LEN), dtype=np.int64)          # all <pad> to start
    for i, text in enumerate(texts):
        ids = [vocab.get(w, 1) for w in tokenize(text)][:MAX_LEN]
        rows[i, MAX_LEN - len(ids):] = ids                            # padding in FRONT
    return torch.from_numpy(rows)

X_train, X_val, X_test = encode(train_df["text"]), encode(val_df["text"]), encode(test_df["text"])
y_train = torch.tensor(train_df["label"].values)
y_val = torch.tensor(val_df["label"].values)
print(f"vocabulary: {len(vocab)} words   X_train {tuple(X_train.shape)}")

vocabulary: 290 words   X_train (3164, 50)


In [7]:
# Build the model that reads reviews in order.
#   1. turn word IDs into 64-number vectors      -> nn.Embedding(vocab_size, 64, padding_idx=0)
#   2. read them in order with a 64-number memory -> cell(64, 64, batch_first=True)
#   3. the head: turn the memory after the LAST word into 2 scores -> nn.Linear(64, 2)
class ReviewReader(nn.Module):
    def __init__(self, vocab_size, cell=nn.LSTM):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, 64, padding_idx=0)
        self.rnn = cell(64, 64, batch_first=True)
        self.head = nn.Linear(64, 2)

    def forward(self, x):
        embedded = self.embedding(x)
        output, _ = self.rnn(embedded)
        last_word_memory = output[:, -1, :]
        return self.head(last_word_memory)

In [8]:
# CHECK — Task 2 model
if ReviewReader is None:
    print("[not yet] Task 2: replace `ReviewReader = None` with your model class.")
else:
    set_seed()
    _m = ReviewReader(len(vocab))
    _out = _m(X_train[:4])
    assert tuple(_out.shape) == (4, 2), f"expected 2 scores per review, got shape {tuple(_out.shape)}"
    _emb = [m for m in _m.modules() if isinstance(m, nn.Embedding)]
    assert _emb and _emb[0].padding_idx == 0, "use nn.Embedding(..., padding_idx=0)"
    assert any(isinstance(m, nn.LSTM) for m in _m.modules()), "the default cell should be nn.LSTM"
    print(f"[ok] Task 2: your model has {sum(p.numel() for p in _m.parameters()):,} learned numbers")

[ok] Task 2: your model has 51,970 learned numbers


In [9]:
# GIVEN — the training loop from Week 3 and the lecture: five lines, validation, early stopping.
# Each batch: predict (2 scores per review) -> measure (cross-entropy against the true label)
#          -> nudge every learned number, the embedding rows included.
def train_reader(model, max_epochs=40, patience=6):
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()
    batches = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)
    best_val, best_weights, waited = 0.0, None, 0
    t0 = time.perf_counter()
    for epoch in range(1, max_epochs + 1):
        model.train()
        for xb, yb in batches:
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            val_acc = (model(X_val).argmax(1) == y_val).float().mean().item()
        if val_acc > best_val:
            best_val, waited = val_acc, 0
            best_weights = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            waited += 1
            if waited >= patience:
                break
    model.load_state_dict(best_weights)
    model.eval()
    return model, time.perf_counter() - t0

def predict(model, X):
    with torch.no_grad():
        return model(X).argmax(1).numpy()

def count(model):
    return sum(p.numel() for p in model.parameters())

In [10]:
# GIVEN — train your LSTM and score it on the test set.
if ReviewReader is None:
    print("[not yet] Task 2: build ReviewReader above, then re-run this cell.")
else:
    set_seed()
    lstm_model, lstm_secs = train_reader(ReviewReader(len(vocab)))
    slice_report("LSTM reader", predict(lstm_model, X_test), lstm_secs, count(lstm_model))
    show_results()

                            overall  simple  contrast  scope_short  scope_long  train seconds  learned numbers
model                                                                                                         
bag of words: single words   0.6453     1.0    0.5000       0.5327      0.5920           0.10              289
bag of words: + word pairs   0.7533     1.0    0.9291       0.5357      0.5987           0.08             1350
LSTM reader                  1.0000     1.0    1.0000       1.0000      1.0000          10.10            51970


---
## Task 3 — Where did reading in order help?

The table above now has three rows. Read it slice by slice.

### Your answer — Task 3

The LSTM ties both bag of words models on simple (1.0000 for all), which is because a simple review has no order dependency, so ignoring order costs nothing there. LSTM wins everywhere else, 1.0000 vs 0.9291 word pairs and 0.5000 single words, on scope_short, 1.0000 vs 0.5357 word pairs and 0.5327 single words, on scope_long, 1.0000 vs 0.5987 word pairs and 0.5920, about 40 to 47 accuracy points over the stronger bag of words baseline on every slice that requires tracking word order or a long distance dependency.

---
## Task 4 — Test it yourself

Write your own test reviews. Each **pair** must use exactly the **same words
in a different order**, so that the meaning flips — like the `contrast`
reviews above. Then see what each model makes of them.

`unknown_words(text)` lists any word the LSTM never saw in training (it would
become `<unk>`); rewrite those, using words from the example reviews.

In [11]:
# GIVEN — helpers for Task 4.
def unknown_words(text):
    return [w for w in tokenize(text) if w not in vocab]

def compare(texts):
    rows = []
    for t in texts:
        with torch.no_grad():
            lstm_p = torch.softmax(lstm_model(encode([t])), 1)[0, 1].item()
        rows.append({
            "review": t[:70] + ("…" if len(t) > 70 else ""),
            "single words": round(float(words_model.predict_proba(words_vec.transform([t]))[0, 1]), 3),
            "+ word pairs": round(float(pairs_model.predict_proba(pairs_vec.transform([t]))[0, 1]), 3),
            "LSTM": round(lstm_p, 3),
        })
    print("probability of POSITIVE from each model")
    print(pd.DataFrame(rows).to_string(index=False))

In [12]:
# Three pairs: (review, the same words re-ordered so the meaning flips).
my_pairs = [
    ("the screen is bright and crisp but the battery drains overnight while it is off",
     "the battery drains overnight while it is off but the screen is bright and crisp"),
    ("delivery arrived a day early but the fit runs two sizes small",
     "the fit runs two sizes small but delivery arrived a day early"),
    ("the build quality feels premium but the companion app crashes on every launch",
     "the companion app crashes on every launch but the build quality feels premium"),
]

In [13]:
# CHECK — Task 4 pairs
if my_pairs is None:
    print("[not yet] Task 4: replace `my_pairs = None` with at least 3 pairs.")
elif "lstm_model" not in globals():
    print("[not yet] Task 4 needs your Task 2 model trained first.")
else:
    assert len(my_pairs) >= 3, "write at least 3 pairs"
    for a, b in my_pairs:
        assert sorted(tokenize(a)) == sorted(tokenize(b)), f"not the same words:\n  {a}\n  {b}"
        assert a != b, "the two reviews in a pair must be in a different order"
        missing = unknown_words(a)
        if missing:
            print(f"   note: the LSTM has never seen {missing} — consider rewording")
    print(f"[ok] Task 4: {len(my_pairs)} valid pairs\n")
    compare([t for pair in my_pairs for t in pair])

[ok] Task 4: 3 valid pairs

probability of POSITIVE from each model
                                                                 review  single words  + word pairs  LSTM
the screen is bright and crisp but the battery drains overnight while …         0.490         0.400 0.000
the battery drains overnight while it is off but the screen is bright …         0.490         0.543 0.999
          delivery arrived a day early but the fit runs two sizes small         0.503         0.373 0.000
          the fit runs two sizes small but delivery arrived a day early         0.503         0.608 0.999
the build quality feels premium but the companion app crashes on every…         0.526         0.407 0.000
the companion app crashes on every launch but the build quality feels …         0.526         0.608 0.999


### Your answer — Task 4

On every pair, the single words model gives the same probability no matter the order (0.490/0.490, 0.503/0.503, 0.526/0.526), reordering the text doesn't change it's feature vector at all. The word pairs model shifts in the right direction but only a little (example, 0.400 vs 0.543 on the first pair) since it only picks up the bigram touching "but," not the entire trailing clause. The LSTM gets every pair right and does so well, 0.000 when the negative clause comes last, 0.999 when the positive clause comes last. The most surprising result is how little confidence the word pairs model has even when it's pointed the right way, 0.373 vs 0.608 on the delivery pair is not good, while the LSTM, which actually reads the sentence in sequence, treats the same case as unambiguous.

---
## Task 5 — Swap the cell: LSTM → GRU

The GRU is the lighter design from the lecture: two gates instead of three.
Train a GRU version of your model — one word changes — and compare accuracy,
size, and training time. The embedding table and the head stay exactly as they
were; only the cell that carries the memory changes.

In [14]:
# Same ReviewReader, with the recurrent cell swapped from LSTM to GRU.
set_seed()
gru_untrained = ReviewReader(len(vocab), cell=nn.GRU)

In [15]:
# CHECK + GIVEN — train and score the GRU.
if gru_untrained is None:
    print("[not yet] Task 5: replace `gru_untrained = None` with your GRU model.")
else:
    assert any(isinstance(m, nn.GRU) for m in gru_untrained.modules()), "use cell=nn.GRU"
    gru_model, gru_secs = train_reader(gru_untrained)
    slice_report("GRU reader", predict(gru_model, X_test), gru_secs, count(gru_model))
    print("[ok] Task 5\n")
    show_results()

[ok] Task 5

                            overall  simple  contrast  scope_short  scope_long  train seconds  learned numbers
model                                                                                                         
bag of words: single words   0.6453  1.0000    0.5000       0.5327      0.5920           0.10              289
bag of words: + word pairs   0.7533  1.0000    0.9291       0.5357      0.5987           0.08             1350
LSTM reader                  1.0000  1.0000    1.0000       1.0000      1.0000          10.10            51970
GRU reader                   0.9992  0.9963    1.0000       1.0000      1.0000          18.71            43650


### Your answer — Task 5

The GRU almost matches the LSTM, 0.9992 overall vs. 1.0000, tying on contrast, scope_short, and scope_long (all 1.0000) but missing a couple of simple reviews (0.9963 vs a perfect 1.0000). It gets there with about 16% less learned numbers (43,650 vs 51,970), and the GRU took longer to run (18.71s vs 10.10s).. I'd ship the LSTM, because the gap is small but significant, the parameter savings aren't decisive at this scale, and it has a faster training time.

---
## Task 6 — The recommendation

### Your answer — Task 6: memo to the product owner

Subject: Recommendation: Replace the bag of words sentiment model with the LSTM

Today's model (TF IDF bag of words) handles straightforward reviews with 100% accuracy, but on reviews built around a contrast like "X but Y," it's accuracy is 50.0%, because it can not tell which half of the sentence the customer meant last. It's also 53.3%–59.9% accurate on reviews that phrase an opinion as a negation ("it is not true that") when the actual opinion sits more than a few words away. Adding two word phrases as features fixes the contrast cases (92.9% accuracy) but not the negation cases (53.6%-59.9%), because the deciding words are too far apart for a two word window to catch.

The alternative built this sprint: an LSTM that reads each review in order and carries memory across the whole sentence, scores 100% on every one of those slices, and 100% overall vs 64.5% (current model) or 75.3% (word-pairs version).

Cost: training takes about 10 seconds on an ordinary laptop CPU, and the model has about 52,000 learned numbers, which is trivial to store or run at our review volume. 

When would this change? Bag of words only catches up if basically all incoming reviews are the simple, single opinion type, which is the one case where both approaches already score 100%. The moment any real share of traffic uses contrast or negation phrasing, which is why support flagged this in the first place, the LSTM's advantage holds. There's effectively no realistic traffic mix where you would keep bag of words.

One limitation: this evaluation used synthetic, template generated reviews, with vocabulary the model already saw during training and only two structural tricks (contrast and negation). Real customers will phrase both in more varied ways and may use words, slang, or misspellings the model has never encountered. Before full rollout, I would measure the LSTM's accuracy on a sample of the actual tickets support has already flagged.